# Unified technical validation

This notebook is the single executable record for the OdorNet technical
validation workflow. It contains the source-level preprocessing audit, SEA
reproducibility checks, label coverage and deletion stability analyses,
stereochemistry-aware leakage audits, controlled hierarchy validation, the raw
multi-dataset comparison, the SEA missing-label integration-mode comparison,
and result visualizations.

The expensive stages are controlled by explicit flags. Existing cached result
files are reused where the component notebook defines caching. The notebook
uses the published `dataset_train_aligned.csv`, `dataset_val_aligned.csv`, and
`dataset_test_aligned.csv` files as the single fixed 7:2:1 OdorNet split while
making the additional raw-data comparisons and integration-policy experiments
directly reproducible.

## Shared training protocol for the comparison experiments

The raw multi-dataset comparison and the SEA integration-mode comparison use
the same model implementations and optimization protocol. Each run is a
12-label or dataset-native multi-label classification task, depending on the
experiment. Models are trained for a maximum of 30 epochs with a batch size of
48 and random seed 959. The training, validation, and test data loaders use
shuffling only for the training split. No learning-rate scheduler, gradient
clipping, or early stopping is used. After every epoch, the checkpoint with the
highest validation Macro F1 at a fixed decision threshold of 0.5 is retained;
the retained checkpoint is then evaluated once on the test split.

The loss is a class-weighted binary cross-entropy with logits. For label j, the
positive-class weight is computed from the training split as
`N_negative,j / N_positive,j`. The `drop` missing-label policy masks unresolved
cells out of both loss and metric calculations. The `union` policy maps
unresolved cells to positive targets, whereas `intersection` maps them to
negative targets. The raw multi-dataset comparison uses `drop`; the SEA
integration-mode experiment varies this policy while holding all other
settings fixed.

For the MolFormer model, a locally stored pretrained MolFormer encoder is
fine-tuned end to end. SMILES are tokenized with truncation and fixed padding
to 256 tokens using the `eager` attention implementation. Masked mean pooling
of the final hidden states is passed through an MLP with dimensions
`768 -> 512 -> 384 -> 256`, LayerNorm and GELU after each hidden layer, and
dropout 0.02, followed by a linear output layer with one logit per target
label. MolFormer uses AdamW with learning rate `1e-5`, weight decay `1e-6`,
and an additional L1 penalty of `1e-8` on the MLP and final classification
layer.

For the graph baseline, RDKit converts each SMILES string into an undirected
molecular graph. Each atom is represented by a 16-dimensional one-hot atom
type vector. The network contains two GCNConv layers with hidden dimension
256, ReLU activations, and dropout 0.2 between graph-convolution layers.
Global mean pooling produces a molecule representation, followed by a
`256 -> 128 -> output` classifier with ReLU and dropout 0.2. The GNN uses
AdamW with learning rate `1e-3` and weight decay `1e-5`.

All runs use full precision and deterministic random-state settings for
Python, NumPy, and PyTorch. The raw comparison preserves native labels for
the external datasets after frequency filtering, whereas the released
OdorNet comparison uses the fixed 12-category SEA label space. Consequently,
the architecture and optimization protocol are matched, while the number of
output logits is dataset-dependent in the raw multi-dataset experiment.

In [ ]:
# Unified execution switches.
# Set these to True when reproducing the corresponding training stages.
RUN_REVIEWER_TRAINING = False
RUN_RAW_COMPARISON_TRAINING = False
RUN_SEA_MODE_TRAINING = False

# The reviewer cells use this name; keeping it here makes the switch global.
RUN_EXPENSIVE_TRAINING = RUN_REVIEWER_TRAINING

print("Reviewer training enabled:", RUN_REVIEWER_TRAINING)
print("Raw comparison training enabled:", RUN_RAW_COMPARISON_TRAINING)
print("SEA integration-mode training enabled:", RUN_SEA_MODE_TRAINING)

In [ ]:
# Repository setup for the technical-validation workflow.
from pathlib import Path
import json
import os
import sys

import numpy as np
import pandas as pd
from IPython.display import display

start_dir = Path.cwd().resolve()
for candidate in (start_dir, *start_dir.parents):
    if (candidate / "scripts" / "run_reviewer_evaluations.py").exists():
        ROOT = candidate
        break
else:
    raise RuntimeError(
        "Could not locate the OdorNet repository root from the current directory."
    )

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "src"))

from scripts.run_reviewer_evaluations import (
    DEFAULT_OUTPUT_DIR,
    DEFAULT_RESULTS_DIR,
    run_controlled_comparison,
    run_hierarchy_randomization,
    run_stereo_safe_split_training,
    run_taxonomy_audits,
)
from odornet.datasets import LABEL_COLUMNS, load_odornet, load_source_metadata
from odornet.reviewer_evaluations import (
    audit_source_smiles_preprocessing,
    build_secondary_assignment_table,
    canonical_structure_keys,
)
from odornet.sea import load_main_label_mapping

RESULTS_DIR = DEFAULT_RESULTS_DIR
OUTPUT_DIR = DEFAULT_OUTPUT_DIR
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_SOURCE_PATH = ROOT / "data" / "raw" / "merged_8892_cleaned_251230.pkl"
print(f"Repository root: {ROOT}")

In [ ]:
# Fixed technical-validation settings. Set the final flag to True only
# when reproducing the full controlled and randomized training runs.
EPOCHS = 30
BATCH_SIZE = 48
TRAINING_SEED = 959
DELETION_REPETITIONS = 10
HIERARCHY_REPETITIONS = 3
# RUN_EXPENSIVE_TRAINING is defined by the unified execution switch above.

TECHNICAL_VALIDATION_METRIC = (
    "validation_optimized_macro_f1_with_per_label_thresholds"
)

## 1. Source-Level RDKit Preprocessing and SMILES Deduplication

This is the reproducible preprocessing audit requested by the reviewer. The
input is the restored source snapshot, whose nested `Source` list contains one
record per source annotation. For every `Original_SMILES`, the procedure is:

1. Treat blank values as unusable input and count them separately.
2. Call `Chem.MolFromSmiles(..., sanitize=False)` to separate parser failure
   from chemical sanitization failure.
3. Call `Chem.SanitizeMol(..., catchErrors=True)` and record the returned RDKit
   sanitization flag when it fails.
4. Convert valid molecules to canonical **isomeric** SMILES with
   `Chem.MolToSmiles(..., canonical=True, isomericSmiles=True)`.
5. Deduplicate only after successful cleaning, using that canonical isomeric
   SMILES as the key; retain the first deterministically sorted source record.

The released snapshot does not include the original upstream dumps or an
historical deletion ledger. Consequently, the counts below report all failures
and removals observable from the released source-record input. They do not
retroactively estimate records removed before this snapshot was created.

In [ ]:
# Technical validation: audit every nested source record before deduplication.
source_df = load_source_metadata(root=ROOT, metadata_path=RAW_SOURCE_PATH)
source_smiles_audit, unique_smiles, preprocessing_summary = (
    audit_source_smiles_preprocessing(source_df)
)

source_smiles_audit.to_csv(
    RESULTS_DIR / "smiles_preprocessing_audit.csv", index=False
)
source_preprocessing_by_source = (
    source_smiles_audit.assign(
        valid_record=source_smiles_audit["status"].eq("valid"),
        removed_record=source_smiles_audit["status"].ne("valid"),
    )
    .groupby("source_id", as_index=False)
    .agg(
        source_record_count=("source_id", "size"),
        valid_record_count=("valid_record", "sum"),
        removed_record_count=("removed_record", "sum"),
        unique_cleaned_smiles=("cleaned_smiles", "nunique"),
    )
)
source_preprocessing_by_source["duplicate_records_removed_within_source"] = (
    source_preprocessing_by_source["valid_record_count"]
    - source_preprocessing_by_source["unique_cleaned_smiles"]
)
source_preprocessing_by_source.to_csv(
    RESULTS_DIR / "smiles_preprocessing_by_source.csv", index=False
)
unique_smiles.to_csv(
    RESULTS_DIR / "unique_cleaned_isomeric_smiles.csv", index=False
)
released_full_for_preprocessing = load_odornet("full", root=ROOT)
released_full_isomeric_keys = {
    canonical_structure_keys(smiles)["canonical_isomeric_smiles"]
    for smiles in released_full_for_preprocessing["SMILES"]
}
preprocessing_summary["cleaned_set_matches_released_full_isomeric_set"] = (
    set(unique_smiles["cleaned_smiles"]) == released_full_isomeric_keys
)
(RESULTS_DIR / "smiles_preprocessing_summary.json").write_text(
    json.dumps(preprocessing_summary, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)

preprocessing_summary_table = pd.DataFrame(
    [
        {"quantity": key, "count_or_value": value}
        for key, value in preprocessing_summary.items()
    ]
)
display(preprocessing_summary_table)
display(
    source_smiles_audit["status"]
    .value_counts()
    .rename_axis("rdkit_status")
    .reset_index(name="record_count")
)
display(source_preprocessing_by_source)

assert preprocessing_summary["rdkit_parse_failures"] == 0
assert preprocessing_summary["rdkit_sanitize_failures"] == 0
assert preprocessing_summary["invalid_or_unusable_records_removed"] == 0
assert preprocessing_summary["unique_cleaned_isomeric_smiles"] == len(unique_smiles)
assert preprocessing_summary["cleaned_set_matches_top_level_set"]
assert preprocessing_summary["cleaned_set_matches_released_full_isomeric_set"]

print(
    f"RDKit-valid source records: {preprocessing_summary['valid_source_records']}"
)
print(
    f"Unique cleaned isomeric SMILES: "
    f"{preprocessing_summary['unique_cleaned_isomeric_smiles']}"
)
print(
    f"Duplicate source records removed after cleaning: "
    f"{preprocessing_summary['duplicate_source_records_removed']}"
)

**Interpretation of the current audit.** The snapshot contains 23,247
source records and 8,892 unique cleaned isomeric SMILES. All 23,247 records
parse and sanitize successfully, so the observable RDKit parse/sanitize failure
count is zero and no invalid record is removed at this stage. Deduplication
removes 14,355 repeated source records. The 8,892 resulting canonical keys
match the 8,892 top-level molecule rows. This does not mean that the upstream
source acquisition process could never have failed; those pre-snapshot records
are not included in the current artifact.

## 2. Stereochemistry-safe 7:2:1 split audit

The released split is checked pairwise for exact stored-SMILES overlap,
canonical-isomeric-SMILES overlap, and canonical-connectivity overlap. A zero
count for all three comparisons ensures that alternate notations and
stereoisomers cannot cross from training into validation or test. The released
validation and test partitions contain only explicit labels, whereas unresolved
training labels are retained for the documented Drop, Union, and Intersection
policies.

In [ ]:
# Technical validation: audit the published 7:2:1 split.
split_frames = {
    "train": load_odornet("train", root=ROOT),
    "val": load_odornet("val", root=ROOT),
    "test": load_odornet("test", root=ROOT),
}

def structure_key_frame(frame, split_name):
    keys = frame["SMILES"].map(canonical_structure_keys).apply(pd.Series)
    keys.insert(0, "SMILES", frame["SMILES"].to_numpy())
    keys.insert(0, "row_index", np.arange(len(frame)))
    keys.insert(0, "split", split_name)
    return keys

key_frames = {
    split: structure_key_frame(frame, split)
    for split, frame in split_frames.items()
}

split_audit_rows = []
for left, right in (("train", "val"), ("train", "test"), ("val", "test")):
    left_keys = key_frames[left]
    right_keys = key_frames[right]
    split_audit_rows.append(
        {
            "left_split": left,
            "right_split": right,
            "raw_stored_smiles_overlap": len(set(left_keys["SMILES"]) & set(right_keys["SMILES"])),
            "canonical_isomeric_overlap": len(
                set(left_keys["canonical_isomeric_smiles"])
                & set(right_keys["canonical_isomeric_smiles"])
            ),
            "connectivity_overlap": len(
                set(left_keys["connectivity_smiles"])
                & set(right_keys["connectivity_smiles"])
            ),
        }
    )

split_audit = pd.DataFrame(split_audit_rows)
nan_summary = pd.DataFrame(
    [
        {
            "split": split,
            "rows": len(frame),
            "nan_cells": int(frame[LABEL_COLUMNS].isna().sum().sum()),
            "nan_rows": int(frame[LABEL_COLUMNS].isna().any(axis=1).sum()),
        }
        for split, frame in split_frames.items()
    ]
)
display(nan_summary)
display(split_audit)
assert list(nan_summary["rows"]) == [6224, 1778, 890]
assert int(nan_summary.loc[nan_summary["split"].isin(["val", "test"]), "nan_cells"].sum()) == 0
assert int(split_audit[["raw_stored_smiles_overlap", "canonical_isomeric_overlap", "connectivity_overlap"]].to_numpy().sum()) == 0

nan_summary.to_csv(RESULTS_DIR / "released_split_nan_summary.csv", index=False)
split_audit.to_csv(RESULTS_DIR / "released_split_structure_audit.csv", index=False)


## 3. SEA Reproducibility: Matrix, Graph, Expert Correction, and AI Alignment

This section makes the SEA implementation details explicit. The statistical
audit uses one transaction per top-level molecule row and treats each
`Processed_Labels` list as a set of descriptors for that transaction. With
binary incidence matrix `X`, the co-occurrence matrix is `C = X.T @ X` and
`C[i, i]` is the observation count for descriptor `i`. The conditional
co-occurrence matrix is `P(j|i) = C[i,j] / C[i,i]`.

The graph visualization uses `min_count = 30` and a directed edge threshold
`P(j|i) >= 0.45`; the layout is NetworkX spring layout with seed 959 and is
only an inspection aid. **No clustering algorithm is used to create the
released taxonomy**, so a clustering method and clustering distance are not
applicable. The semantic audit later in this notebook uses Sentence
Transformer embeddings and cosine similarity, but that is a consistency
measurement rather than a clustering step.

In [ ]:
# Technical validation: regenerate and save the complete S-stage matrices.
transactions = [sorted(set(labels)) for labels in source_df["Processed_Labels"]]
all_terms = sorted({term for labels in transactions for term in labels})
incidence = pd.DataFrame(
    0, index=np.arange(len(transactions)), columns=all_terms, dtype=np.int32
)
for row_index, labels in enumerate(transactions):
    incidence.loc[row_index, labels] = 1

cooccurrence_counts = incidence.T @ incidence
term_counts = pd.Series(
    np.diag(cooccurrence_counts), index=cooccurrence_counts.index, name="count"
)
cooccurrence_probability = cooccurrence_counts.div(
    term_counts.replace(0, np.nan), axis=0
).fillna(0.0)
cooccurrence_counts.to_csv(RESULTS_DIR / "sea_cooccurrence_counts.csv")
cooccurrence_probability.to_csv(RESULTS_DIR / "sea_cooccurrence_probability.csv")

MIN_DESCRIPTOR_COUNT = 30
MIN_CONDITIONAL_CONFIDENCE = 0.45
statistical_edges = []
for antecedent in cooccurrence_probability.index:
    if term_counts[antecedent] < MIN_DESCRIPTOR_COUNT:
        continue
    for consequent in cooccurrence_probability.columns:
        if antecedent == consequent or term_counts[consequent] < MIN_DESCRIPTOR_COUNT:
            continue
        confidence = float(cooccurrence_probability.loc[antecedent, consequent])
        if confidence >= MIN_CONDITIONAL_CONFIDENCE:
            statistical_edges.append(
                {
                    "antecedent": antecedent,
                    "consequent": consequent,
                    "conditional_probability": confidence,
                }
            )
statistical_edges = pd.DataFrame(statistical_edges)
statistical_edges.to_csv(RESULTS_DIR / "sea_statistical_threshold_edges.csv", index=False)

sea_statistical_specification = pd.DataFrame(
    [
        {"parameter": "transaction_unit", "value": "one top-level molecule row"},
        {"parameter": "incidence_values", "value": "binary; duplicate descriptors within a transaction count once"},
        {"parameter": "cooccurrence_matrix", "value": "C = X.T @ X"},
        {"parameter": "conditional_probability", "value": "P(B|A) = C[A,B] / C[A,A]"},
        {"parameter": "minimum_descriptor_count", "value": MIN_DESCRIPTOR_COUNT},
        {"parameter": "minimum_directed_edge_confidence", "value": MIN_CONDITIONAL_CONFIDENCE},
        {"parameter": "clustering_method", "value": "none; graph is descriptive only"},
        {"parameter": "clustering_distance", "value": "not applicable"},
        {"parameter": "graph_layout", "value": "NetworkX spring_layout(seed=959)"},
    ]
)
display(sea_statistical_specification)
print(
    f"S-stage transactions: {len(transactions)}, descriptors: {len(all_terms)}, "
    f"thresholded directed edges: {len(statistical_edges)}"
)

### Expert Correction (E)

The released `final_specialist_label_mapping.json` is a curated correction
layer. It contains 190 descriptor-category relations across eight primary
categories. In the deterministic release code, the statistical/AI mapping is
loaded first, `odorless` is added explicitly, and expert descriptors are
appended to matching primary categories. Each category list is then converted
to a sorted set, so repeated relations are removed while a descriptor can
remain assigned to multiple primary categories. The expert file is therefore a
correction/extension layer, not a replacement for the statistical/AI mapping.

The exact expert review form and reviewer-by-reviewer decision log are not in
the released artifact. The JSON mapping is the reproducible correction input;
the notebook reports that limitation rather than inventing an unavailable
inter-rater protocol.

In [ ]:
# Technical validation: inspect the released E-layer and merged mapping sizes.
with (ROOT / "data" / "metadata" / "final_specialist_label_mapping.json").open(
    "r", encoding="utf-8"
) as handle:
    specialist_mapping = json.load(handle)
with (ROOT / "data" / "metadata" / "olfactory_classification_strong_weak.json").open(
    "r", encoding="utf-8"
) as handle:
    ai_mapping = json.load(handle)

expert_relation_summary = pd.DataFrame(
    [
        {
            "primary_category": category,
            "expert_relation_count": len(terms),
            "expert_unique_descriptor_count": len(set(terms)),
        }
        for category, terms in sorted(specialist_mapping.items())
    ]
)
display(expert_relation_summary)
print(
    "Expert relations:", sum(len(terms) for terms in specialist_mapping.values()),
    "unique expert descriptors:", len(
        {term for terms in specialist_mapping.values() for term in terms}
    ),
)

released_mapping = load_main_label_mapping()
released_assignments = build_secondary_assignment_table()
print(
    "Merged released mapping relations:",
    int(released_assignments["primary_category"].ne("").sum()),
    "including explicit excluded assignments:",
    int(released_assignments["assignment_tier"].eq("excluded").sum()),
)

### AI-Assisted Alignment (A)

The AI alignment input is `olfactory_data_translated.json`: one English
definition per descriptor where available. The historical survey prompt asks
the model to classify using only that English description and supplies the 11
odor categories plus `not any type`. The voting script selected three model
columns and three repeated responses per model, giving nine votes per
descriptor. A category is **strong** at at least eight votes, **weak** at three
to seven votes, and a descriptor with no category reaching three votes is put
in `not any type`. The released JSON retains the resulting strong/weak lists.

The external model calls and raw response workbook are not part of this
repository, so the external voting itself is not rerunnable offline. The
released mapping, prompt logic, vote thresholds, expert layer, and deterministic
label aggregation are rerunnable and are all recorded here. Semantic
consistency is independently audited below with
`sentence-transformers/all-MiniLM-L6-v2`, normalized embeddings, and cosine
similarity.

In [ ]:
# Technical validation: record the A-layer vote protocol and released counts.
ai_protocol = pd.DataFrame(
    [
        {"parameter": "description_input", "value": "English definitions in olfactory_data_translated.json"},
        {"parameter": "model_columns_selected", "value": "gemini-3-pro; gpt-5.2; grok_4_fast"},
        {"parameter": "repeated_votes_per_model", "value": 3},
        {"parameter": "votes_per_descriptor", "value": 9},
        {"parameter": "strong_threshold", "value": ">= 8 votes"},
        {"parameter": "weak_threshold", "value": "3-7 votes"},
        {"parameter": "no_assignment", "value": "no category reaches 3 votes -> not any type"},
        {"parameter": "category_matching", "value": "historical script uses category-string containment"},
    ]
)
display(ai_protocol)
ai_counts = pd.DataFrame(
    [
        {
            "primary_category": category,
            "strong_descriptor_count": len(set(ai_mapping.get(category, []))),
            "weak_descriptor_count": len(set(ai_mapping.get(f"{category}_weak", []))),
        }
        for category in LABEL_COLUMNS
        if category != "odorless"
    ]
)
display(ai_counts)
print("Released not-any-type descriptor count:", len(ai_mapping.get("not any type", [])))

## 4. Assignment Coverage

Coverage is measured against two observed source descriptor scopes: the
molecule-level `Processed_Labels` list and the fully expanded source-record
`Processed_Labels` lists. A descriptor may receive a single primary
assignment, multiple primary assignments, an explicit `not any type` discard,
or no released assignment.

In [ ]:
# This call also regenerates the 20% deletion experiment in the next
# section, keeping both taxonomy validations tied to the same released inputs.
run_taxonomy_audits(
    results_dir=RESULTS_DIR,
    deletion_repetitions=DELETION_REPETITIONS,
)

coverage = pd.read_csv(RESULTS_DIR / "label_coverage_summary.csv")
display(coverage)

## 5. SEA Stability After Removing 20% of Source Labels

For each deterministic repetition, exactly 20% of `Processed_Labels` instances
are removed without replacement. The released SEA mapping and Double-Drop
aggregation are then applied again. This quantifies sensitivity to source-label
perturbation conditional on the released taxonomy; it does not rerun the
unavailable external AI voting process.

In [ ]:
# Per-repetition agreement values are kept for reproducibility rather
# than being compressed into a single mean before inspection.
deletion_runs = pd.read_csv(
    RESULTS_DIR / "sea_label_deletion_stability_repetitions.csv"
)
display(deletion_runs)

deletion_summary = pd.read_json(
    RESULTS_DIR / "sea_label_deletion_stability_summary.json",
    typ="series",
)
display(deletion_summary.to_frame("value"))

## 9. Raw multi-dataset comparison

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from odornet.datasets import LABEL_COLUMNS as ODORNET_LABELS
from odornet.raw_comparison import clean_source_labels, load_raw_source, split_connectivity_balanced
from odornet.training import TrainingConfig, train_gnn_baseline, train_molformer_baseline

RAW_DIR = ROOT / "data" / "raw"
PROVENANCE_SOURCE_RECORDS = ROOT / "data" / "provenance" / "source_records.csv"
ODORNET_RAW_LABELS_PATH = RAW_DIR / "odornet_source_records_processed_labels.csv"
FIXED_ODORNET_DIR = ROOT / "data" / "processed"
RESULTS_DIR = ROOT / "results" / "reviewer_revision" / "raw_multi_dataset_comparison"
OUTPUT_DIR = ROOT / "outputs" / "reviewer_revision" / "raw_multi_dataset_comparison"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 959
RARE_LABEL_FRACTION = 0.01
SPLIT_SEED_CANDIDATES = tuple(range(4))
TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.7, 0.2, 0.1
THRESHOLD = 0.5
# RUN_RAW_COMPARISON_TRAINING is defined by the unified switch above.

DATASET_SPECS = {
    "odornet_raw": {
        "path": ODORNET_RAW_LABELS_PATH,
        "smiles_column": "SMILES",
        "label_column": "Processed_Labels",
    },
    "gs_lf": {"path": RAW_DIR / "gs_lf.csv", "smiles_column": "IsomericSMILES"},
    "smiles_to_smell": {"path": RAW_DIR / "smiles_to_smell.csv", "smiles_column": "SMILES"},
    "arctander": {"path": RAW_DIR / "arctander.csv", "smiles_column": "SMILES"},
}

print("Repository root:", ROOT)
print("Training enabled:", RUN_RAW_COMPARISON_TRAINING)

## 1. Project OdorNet raw labels and clean all raw datasets

In [ ]:
odornet_raw_projection = pd.read_csv(
    PROVENANCE_SOURCE_RECORDS,
    usecols=["SMILES", "Processed_Labels"],
)
odornet_raw_projection.to_csv(ODORNET_RAW_LABELS_PATH, index=False)
print(f"Projected {len(odornet_raw_projection)} OdorNet source records to {ODORNET_RAW_LABELS_PATH}")

source_frames = {}
source_summaries = {}

for dataset_name, spec in DATASET_SPECS.items():
    raw_frame = load_raw_source(
        spec["path"],
        spec["smiles_column"],
        label_column=spec.get("label_column", "Labels"),
    )
    cleaned, summary = clean_source_labels(
        raw_frame,
        dataset_name=dataset_name,
        rare_fraction=RARE_LABEL_FRACTION,
    )
    source_frames[dataset_name] = cleaned
    source_summaries[dataset_name] = summary
    cleaned.to_csv(RESULTS_DIR / f"{dataset_name}_cleaned.csv", index=False)

summary_table = pd.DataFrame(
    [
        {
            "dataset": name,
            "molecules": info["molecule_count"],
            "raw_labels": info["raw_label_count"],
            "kept_labels": info["kept_label_count"],
            "removed_labels": info["removed_label_count"],
            "minimum_count": info["minimum_count_for_retention"],
            "parse_failures": info["parse_failures"],
        }
        for name, info in source_summaries.items()
    ]
)
display(summary_table)
summary_table.to_csv(RESULTS_DIR / "cleaning_summary.csv", index=False)

## 2. Connectivity-aware 7:2:1 split

In [ ]:
prepared = {}

for dataset_name, frame in source_frames.items():
    labels = list(source_summaries[dataset_name]["kept_labels"])
    train, val, test, metadata, balance = split_connectivity_balanced(
        frame,
        labels,
        train_ratio=TRAIN_RATIO,
        val_ratio=VAL_RATIO,
        test_ratio=TEST_RATIO,
        seed_candidates=SPLIT_SEED_CANDIDATES,
    )
    prepared[dataset_name] = {
        "labels": labels,
        "train": train,
        "val": val,
        "test": test,
        "metadata": metadata,
        "balance": balance,
    }
    split_dir = RESULTS_DIR / "splits" / dataset_name
    split_dir.mkdir(parents=True, exist_ok=True)
    for split_name, split_frame in (("train", train), ("val", val), ("test", test)):
        split_frame[["SMILES", *labels]].to_csv(split_dir / f"{split_name}.csv", index=False)
    balance.to_csv(RESULTS_DIR / f"{dataset_name}_label_balance.csv", index=False)
    (RESULTS_DIR / f"{dataset_name}_split_summary.json").write_text(
        json.dumps(metadata, indent=2) + "\n",
        encoding="utf-8",
    )

external_split_overview = pd.DataFrame(
    [
        {
            "dataset": name,
            "labels": len(info["labels"]),
            **info["metadata"]["actual_rows"],
            "size_error": info["metadata"]["size_error"],
            "pairwise_mean_abs_rate_difference": info["metadata"]["pairwise_mean_abs_rate_difference"],
            "max_pairwise_abs_rate_difference": info["metadata"]["max_pairwise_abs_rate_difference"],
            "train_val_mean_abs_rate_difference": info["metadata"]["train_val_mean_abs_rate_difference"],
            "train_test_mean_abs_rate_difference": info["metadata"]["train_test_mean_abs_rate_difference"],
        }
        for name, info in prepared.items()
    ]
)
display(external_split_overview)
external_split_overview.to_csv(RESULTS_DIR / "external_split_overview.csv", index=False)

## 3. Reuse the fixed SEA-based OdorNet split

In [ ]:
odornet = {
    "labels": list(ODORNET_LABELS),
    "train": pd.read_csv(FIXED_ODORNET_DIR / "dataset_train_aligned.csv"),
    "val": pd.read_csv(FIXED_ODORNET_DIR / "dataset_val_aligned.csv"),
    "test": pd.read_csv(FIXED_ODORNET_DIR / "dataset_test_aligned.csv"),
}
assert int(odornet["val"][ODORNET_LABELS].isna().sum().sum()) == 0
assert int(odornet["test"][ODORNET_LABELS].isna().sum().sum()) == 0
print({split: len(odornet[split]) for split in ("train", "val", "test")})

## 4. Split audit

In [ ]:
audit_rows = []
for dataset_name, info in prepared.items():
    labels = info["labels"]
    parts = [info["train"], info["val"], info["test"]]
    key_sets = [set(part["connectivity_smiles"]) for part in parts]
    audit_rows.append(
        {
            "dataset": dataset_name,
            "rows": sum(len(part) for part in parts),
            "train_val_shared_connectivity": len(key_sets[0] & key_sets[1]),
            "train_test_shared_connectivity": len(key_sets[0] & key_sets[2]),
            "val_test_shared_connectivity": len(key_sets[1] & key_sets[2]),
            "val_nan_cells": int(parts[1][labels].isna().sum().sum()),
            "test_nan_cells": int(parts[2][labels].isna().sum().sum()),
        }
    )
audit_df = pd.DataFrame(audit_rows)
display(audit_df)
assert (audit_df[["train_val_shared_connectivity", "train_test_shared_connectivity", "val_test_shared_connectivity"]] == 0).all().all()
assert (audit_df[["val_nan_cells", "test_nan_cells"]] == 0).all().all()
audit_df.to_csv(RESULTS_DIR / "external_split_audit.csv", index=False)

## 5. Train the two baseline models

In [ ]:
def make_config(output_dir: Path) -> TrainingConfig:
    return TrainingConfig(
        nan_policy="drop",
        batch_size=48,
        num_epochs=30,
        threshold=THRESHOLD,
        seed=SEED,
        output_dir=str(output_dir),
        molformer_max_length=256,
        molformer_padding="max_length",
        molformer_attn_implementation="eager",
        l1_lambda=1e-8,
        mixed_precision=False,
        optimize_validation_thresholds=False,
        gnn_num_layers=2,
        verbose=False,
    )

LOCAL_MOLFORMER_PATH = ROOT / "molformer_config"
if not LOCAL_MOLFORMER_PATH.exists():
    LOCAL_MOLFORMER_PATH = None

def run_one_model(dataset_name: str, data: dict[str, object], model_name: str) -> dict[str, object]:
    labels = list(data["labels"])
    train = data["train"][["SMILES", *labels]]
    val = data["val"][["SMILES", *labels]]
    test = data["test"][["SMILES", *labels]]
    output_dir = OUTPUT_DIR / dataset_name / model_name
    config = make_config(output_dir)
    if model_name == "gnn":
        result = train_gnn_baseline(train, val, config=config, labels=labels, test_df=test)
    else:
        result = train_molformer_baseline(
            train,
            val,
            config=config,
            labels=labels,
            local_model_path=LOCAL_MOLFORMER_PATH,
            test_df=test,
        )
    test_metrics = result.get("test_metrics", {})
    return {
        "dataset": dataset_name,
        "model": model_name,
        "best_epoch": int(result.get("best_epoch", -1)),
        "val_macro_f1": float(result.get("best_macro_f1", float("nan"))),
        "test_macro_f1": float(test_metrics.get("test_macro_f1", float("nan"))),
        "test_micro_f1": float(test_metrics.get("test_micro_f1", float("nan"))),
        "test_macro_auroc": float(test_metrics.get("test_macro_auroc", float("nan"))),
        "test_loss": float(test_metrics.get("loss", float("nan"))),
        "threshold": THRESHOLD,
        "optimize_validation_thresholds": False,
        "output_dir": str(result.get("output_dir", output_dir)),
    }

datasets_for_training = {"odornet": odornet, **prepared}
training_rows = []
if RUN_RAW_COMPARISON_TRAINING:
    for dataset_name, data in datasets_for_training.items():
        for model_name in ("gnn", "molformer"):
            cache_path = RESULTS_DIR / f"{dataset_name}_{model_name}_metrics.json"
            if cache_path.exists():
                row = json.loads(cache_path.read_text(encoding="utf-8"))
            else:
                row = run_one_model(dataset_name, data, model_name)
                cache_path.write_text(
                    json.dumps(row, indent=2, allow_nan=True) + "\n",
                    encoding="utf-8",
                )
            training_rows.append(row)
    training_metrics = pd.DataFrame(training_rows)
    training_metrics.to_csv(RESULTS_DIR / "training_metrics.csv", index=False)
    display(training_metrics)
else:
    print("RUN_RAW_COMPARISON_TRAINING=False; training skipped.")

## 6. Save reproducibility manifest

In [ ]:
manifest = {
    "seed": SEED,
    "rare_label_fraction": RARE_LABEL_FRACTION,
    "split_ratios": {"train": TRAIN_RATIO, "val": VAL_RATIO, "test": TEST_RATIO},
    "split_seed_candidates": list(SPLIT_SEED_CANDIDATES),
    "threshold": THRESHOLD,
    "optimize_validation_thresholds": False,
    "datasets": source_summaries,
}
(RESULTS_DIR / "manifest.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)
print("Saved results under", RESULTS_DIR)

## 9b. SEA integration-mode comparison

In [ ]:
from __future__ import annotations

import hashlib
import json
import platform
import sys
from pathlib import Path

import pandas as pd
import torch
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from odornet.datasets import LABEL_COLUMNS
from odornet.reviewer_evaluations import canonical_structure_keys
from odornet.training import TrainingConfig, train_gnn_baseline, train_molformer_baseline

FIXED_SPLIT_DIR = ROOT / "data" / "processed"
RESULTS_DIR = ROOT / "results" / "reviewer_revision" / "sea_integration_modes"
OUTPUT_DIR = ROOT / "outputs" / "reviewer_revision" / "sea_integration_modes"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAINING_SEED = 959
THRESHOLD = 0.5
BATCH_SIZE = 48
NUM_EPOCHS = 30
MODES = ("drop", "union", "intersection")
MODELS = ("gnn", "molformer")
# RUN_SEA_MODE_TRAINING is defined by the unified switch above.

print("Repository root:", ROOT)
print("Training seed:", TRAINING_SEED)
print("Modes:", MODES)
print("Training enabled:", RUN_SEA_MODE_TRAINING)

## 1. Load and audit the fixed SEA split

In [ ]:
split_frames = {
    "train": pd.read_csv(FIXED_SPLIT_DIR / "dataset_train_aligned.csv"),
    "val": pd.read_csv(FIXED_SPLIT_DIR / "dataset_val_aligned.csv"),
    "test": pd.read_csv(FIXED_SPLIT_DIR / "dataset_test_aligned.csv"),
}
for split, frame in split_frames.items():
    missing = set(LABEL_COLUMNS) - set(frame.columns)
    if missing:
        raise ValueError(f"{split} is missing SEA labels: {sorted(missing)}")

nan_summary = pd.DataFrame([
    {
        "split": split,
        "rows": len(frame),
        "nan_cells": int(frame[LABEL_COLUMNS].isna().sum().sum()),
        "nan_rows": int(frame[LABEL_COLUMNS].isna().any(axis=1).sum()),
    }
    for split, frame in split_frames.items()
])
display(nan_summary)
assert int(nan_summary.loc[nan_summary["split"].isin(["val", "test"]), "nan_cells"].sum()) == 0

connectivity = {
    split: set(
        frame["SMILES"].map(canonical_structure_keys).map(lambda item: item["connectivity_smiles"])
    )
    for split, frame in split_frames.items()
}
split_audit = pd.DataFrame([{
    "train_val_shared_connectivity": len(connectivity["train"] & connectivity["val"]),
    "train_test_shared_connectivity": len(connectivity["train"] & connectivity["test"]),
    "val_test_shared_connectivity": len(connectivity["val"] & connectivity["test"]),
}])
display(split_audit)
assert int(split_audit.iloc[0].sum()) == 0
nan_summary.to_csv(RESULTS_DIR / "split_nan_summary.csv", index=False)
split_audit.to_csv(RESULTS_DIR / "split_connectivity_audit.csv", index=False)

## 2. Shared training configuration

In [ ]:
def make_config(mode: str, model_name: str) -> TrainingConfig:
    return TrainingConfig(
        nan_policy=mode,
        batch_size=BATCH_SIZE,
        num_epochs=NUM_EPOCHS,
        threshold=THRESHOLD,
        seed=TRAINING_SEED,
        output_dir=str(OUTPUT_DIR / mode / model_name),
        molformer_max_length=256,
        molformer_padding="max_length",
        molformer_attn_implementation="eager",
        l1_lambda=1e-8,
        mixed_precision=False,
        optimize_validation_thresholds=False,
        gnn_num_layers=2,
        verbose=False,
    )

LOCAL_MOLFORMER_PATH = ROOT / "molformer_config"
if not LOCAL_MOLFORMER_PATH.exists():
    LOCAL_MOLFORMER_PATH = None

train_df = split_frames["train"][["SMILES", *LABEL_COLUMNS]]
val_df = split_frames["val"][["SMILES", *LABEL_COLUMNS]]
test_df = split_frames["test"][["SMILES", *LABEL_COLUMNS]]

## 3. Train six mode/model combinations

In [ ]:
def run_one(mode: str, model_name: str) -> dict[str, object]:
    cache_path = RESULTS_DIR / f"{mode}_{model_name}_metrics.json"
    if cache_path.exists():
        return json.loads(cache_path.read_text(encoding="utf-8"))

    config = make_config(mode, model_name)
    if model_name == "gnn":
        result = train_gnn_baseline(train_df, val_df, config=config, labels=list(LABEL_COLUMNS), test_df=test_df)
    else:
        result = train_molformer_baseline(
            train_df,
            val_df,
            config=config,
            labels=list(LABEL_COLUMNS),
            local_model_path=LOCAL_MOLFORMER_PATH,
            test_df=test_df,
        )

    test_metrics = result.get("test_metrics", {})
    row = {
        "integration_mode": mode,
        "model": model_name,
        "training_seed": TRAINING_SEED,
        "best_epoch": int(result.get("best_epoch", -1)),
        "val_macro_f1": float(result.get("best_macro_f1", float("nan"))),
        "test_macro_f1": float(test_metrics.get("test_macro_f1", float("nan"))),
        "test_micro_f1": float(test_metrics.get("test_micro_f1", float("nan"))),
        "test_macro_auroc": float(test_metrics.get("test_macro_auroc", float("nan"))),
        "test_loss": float(test_metrics.get("loss", float("nan"))),
        "threshold": THRESHOLD,
        "optimize_validation_thresholds": False,
        "batch_size": BATCH_SIZE,
        "num_epochs": NUM_EPOCHS,
        "gnn_num_layers": 2,
        "model_source": str(result.get("model_source", "")),
        "output_dir": str(result.get("output_dir", OUTPUT_DIR / mode / model_name)),
    }
    cache_path.write_text(json.dumps(row, indent=2, allow_nan=True) + "\n", encoding="utf-8")
    return row

rows = []
if RUN_SEA_MODE_TRAINING:
    for mode in MODES:
        for model_name in MODELS:
            rows.append(run_one(mode, model_name))
    metrics = pd.DataFrame(rows).sort_values(["integration_mode", "model"])
    metrics.to_csv(RESULTS_DIR / "sea_integration_modes_metrics.csv", index=False)
    display(metrics)
else:
    print("RUN_SEA_MODE_TRAINING=False; training skipped.")

## 4. Reproducibility manifest

In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

manifest = {
    "experiment": "sea_integration_modes",
    "source_split_directory": str(FIXED_SPLIT_DIR),
    "split_files": {
        split: {
            "path": str(
                FIXED_SPLIT_DIR
                / {
                    "train": "dataset_train_aligned.csv",
                    "val": "dataset_val_aligned.csv",
                    "test": "dataset_test_aligned.csv",
                }[split]
            ),
            "sha256": sha256_file(
                FIXED_SPLIT_DIR
                / {
                    "train": "dataset_train_aligned.csv",
                    "val": "dataset_val_aligned.csv",
                    "test": "dataset_test_aligned.csv",
                }[split]
            ),
            "rows": int(len(split_frames[split])),
        }
        for split in ("train", "val", "test")
    },
    "integration_modes": list(MODES),
    "models": list(MODELS),
    "training_seed": TRAINING_SEED,
    "threshold": THRESHOLD,
    "optimize_validation_thresholds": False,
    "batch_size": BATCH_SIZE,
    "num_epochs": NUM_EPOCHS,
    "gnn_num_layers": 2,
    "nan_policy_semantics": {
        "drop": "ignore unresolved label cells in the loss",
        "union": "treat unresolved label cells as positive",
        "intersection": "treat unresolved label cells as negative",
    },
    "determinism": {
        "python_random_numpy_torch_seed": TRAINING_SEED,
        "cudnn_deterministic": bool(torch.backends.cudnn.deterministic),
        "cudnn_benchmark": bool(torch.backends.cudnn.benchmark),
        "torch_deterministic_algorithms": True,
        "data_loader_workers": 0,
    },
    "software": {
        "python": platform.python_version(),
        "pandas": pd.__version__,
        "torch": torch.__version__,
        "cuda_available": bool(torch.cuda.is_available()),
        "cuda_version": torch.version.cuda,
        "platform": platform.platform(),
    },
}
manifest["source_split_directory"] = "data/processed"
for _split, _entry in manifest["split_files"].items():
    _entry["path"] = f"data/processed/dataset_{_split}_aligned.csv"
(RESULTS_DIR / "reproducibility_manifest.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"
)
print("Saved results under", RESULTS_DIR)

## 10. Result visualizations

The following cells load cached summary tables and epoch logs when available.
They save publication-oriented figures under
`results/reviewer_revision/figures/` and skip plots whose source results are
not yet present.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

FIGURES_DIR = ROOT / "results" / "reviewer_revision" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def save_barplot(frame: pd.DataFrame, x: str, y: str, hue: str, title: str, path: Path) -> None:
    if frame.empty:
        return
    groups = list(frame[x].astype(str).unique())
    hues = list(frame[hue].astype(str).unique())
    positions = np.arange(len(groups))
    width = 0.8 / max(len(hues), 1)
    fig, axis = plt.subplots(figsize=(11, 5))
    for index, hue_value in enumerate(hues):
        values = []
        for group in groups:
            match = frame[(frame[x].astype(str) == group) & (frame[hue].astype(str) == hue_value)]
            values.append(float(match.iloc[0][y]) if not match.empty else np.nan)
        axis.bar(positions + (index - (len(hues) - 1) / 2) * width, values, width, label=hue_value)
    axis.set_xticks(positions, groups, rotation=35, ha="right")
    axis.set_ylabel(y.replace("_", " ").title())
    axis.set_title(title)
    axis.legend(frameon=False)
    axis.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    fig.savefig(path, dpi=240, bbox_inches="tight")
    plt.show()
    plt.close(fig)


raw_metrics_path = ROOT / "results" / "reviewer_revision" / "raw_multi_dataset_comparison" / "training_metrics.csv"
if raw_metrics_path.exists():
    raw_metrics = pd.read_csv(raw_metrics_path)
    raw_metrics["dataset_display"] = raw_metrics["dataset"].map(
        {
            "odornet": "OdorNet SEA",
            "odornet_raw": "OdorNet raw labels",
            "gs_lf": "GS_LF",
            "smiles_to_smell": "SMILES to Smell",
            "arctander": "Arctander",
        }
    ).fillna(raw_metrics["dataset"])
    display(raw_metrics)
    save_barplot(
        raw_metrics,
        x="dataset_display",
        y="test_macro_f1",
        hue="model",
        title="Raw multi-dataset comparison",
        path=FIGURES_DIR / "raw_multi_dataset_test_macro_f1.png",
    )
else:
    print("Raw comparison metrics are not available yet:", raw_metrics_path)


sea_metrics_path = ROOT / "results" / "reviewer_revision" / "sea_integration_modes" / "sea_integration_modes_metrics.csv"
if sea_metrics_path.exists():
    sea_metrics = pd.read_csv(sea_metrics_path)
    display(sea_metrics)
    save_barplot(
        sea_metrics,
        x="integration_mode",
        y="test_macro_f1",
        hue="model",
        title="SEA missing-label integration-mode comparison",
        path=FIGURES_DIR / "sea_integration_mode_test_macro_f1.png",
    )
else:
    print("SEA integration-mode metrics are not available yet:", sea_metrics_path)


controlled_path = ROOT / "results" / "reviewer_revision" / "controlled_comparison_metrics.csv"
if controlled_path.exists():
    controlled = pd.read_csv(controlled_path)
    display(controlled)
    save_barplot(
        controlled,
        x="dataset",
        y="macro_f1",
        hue="model",
        title="Controlled technical-validation comparison",
        path=FIGURES_DIR / "controlled_comparison_macro_f1.png",
    )


def plot_epoch_logs(root: Path, path: Path, title: str) -> None:
    log_paths = sorted(root.rglob("training_logs.json")) if root.exists() else []
    if not log_paths:
        print("No training logs found under:", root)
        return
    fig, axis = plt.subplots(figsize=(12, 6))
    plotted = 0
    for log_path in log_paths:
        try:
            payload = json.loads(log_path.read_text(encoding="utf-8"))
            logs = pd.DataFrame(payload.get("logs", []))
        except (OSError, json.JSONDecodeError, TypeError):
            continue
        if logs.empty or "epoch" not in logs or "val_macro_f1" not in logs:
            continue
        relative = log_path.relative_to(root).parts
        label = "/".join(relative[:-1])
        axis.plot(logs["epoch"], logs["val_macro_f1"], linewidth=1.4, label=label)
        plotted += 1
    if not plotted:
        plt.close(fig)
        print("No compatible validation curves found under:", root)
        return
    axis.set_xlabel("Epoch")
    axis.set_ylabel("Validation Macro F1 at threshold 0.5")
    axis.set_title(title)
    axis.grid(alpha=0.25)
    axis.legend(frameon=False, fontsize=7, ncol=2)
    fig.tight_layout()
    fig.savefig(path, dpi=240, bbox_inches="tight")
    plt.show()
    plt.close(fig)


plot_epoch_logs(
    ROOT / "outputs" / "reviewer_revision" / "raw_multi_dataset_comparison",
    FIGURES_DIR / "raw_multi_dataset_validation_curves.png",
    "Raw multi-dataset validation curves",
)
plot_epoch_logs(
    ROOT / "outputs" / "reviewer_revision" / "sea_integration_modes",
    FIGURES_DIR / "sea_integration_mode_validation_curves.png",
    "SEA integration-mode validation curves",
)